# 09C – REST API with FastAPI (Enterprise)

Build a production-ready REST API for bankruptcy prediction.

## Business Objective
Serve predictions from the trained model to external applications using FastAPI.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib
import uvicorn

MODEL_PATH='production_bankruptcy_model.joblib'
model=joblib.load(MODEL_PATH)

app=FastAPI(title='Bankruptcy Risk Prediction API',version='1.0.0')


In [ ]:
df=pd.read_csv('american_bankruptcy_cleaned.csv')
target='status_label' if 'status_label' in df.columns else 'target'
features=[c for c in df.columns if c!=target]

PredictionRequest=type(
    'PredictionRequest',
    (BaseModel,),
    {'__annotations__':{c:float for c in features}}
)


In [ ]:
@app.get('/')
def home():
    return {'message':'Bankruptcy Risk Prediction API','status':'running'}

@app.get('/health')
def health():
    return {'status':'healthy'}

@app.post('/predict')
def predict(request: PredictionRequest):
    X=pd.DataFrame([request.model_dump()])
    pred=int(model.predict(X)[0])
    prob=float(model.predict_proba(X)[0][1])
    return {
        'prediction':pred,
        'label':'Bankrupt' if pred else 'Healthy',
        'bankruptcy_probability':round(prob,4)
    }


In [ ]:
if __name__=='__main__':
    uvicorn.run(app,host='0.0.0.0',port=8000)


## Endpoints

- GET /
- GET /health
- POST /predict

## Deliverables

- FastAPI service
- Health check endpoint
- Prediction endpoint
- JSON responses
- Ready for Docker deployment
